# LeetCode 1353 — Maximum Number of Events That Can Be Attended bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-events-that-can-be-attended/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def maxEvents(self, events: list[list[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra bài **lập lịch (scheduling)**: mỗi sự kiện chỉ cần **một ngày bất kỳ** trong đoạn `[start, end]`, mỗi ngày dự được tối đa một sự kiện.
2. Hiểu vì sao tham lam "**sắp theo ngày bắt đầu**" là **sai**, còn tham lam "**mỗi ngày chọn sự kiện kết thúc sớm nhất**" là **đúng** (lập luận đổi chỗ — exchange argument).
3. Cài đặt bằng **quét theo ngày + min-heap** (`heapq`) — lời giải chuẩn `O((n + D) log n)`.
4. Biết cách tương đương: **sắp theo ngày kết thúc**, gán **ngày trống sớm nhất ≥ start** (duyệt tuyến tính, rồi tăng tốc bằng **Union-Find "ngày trống kế tiếp"**).
5. Kiểm chứng tham lam bằng **backtracking** (n nhỏ) và **ghép cặp hai phía** (bipartite matching).

## 2. Đề bài & ví dụ

Cho mảng `events`, trong đó `events[i] = [startDay_i, endDay_i]`. Sự kiện `i` có thể được tham dự vào **bất kỳ một ngày** `d` với `startDay_i ≤ d ≤ endDay_i`. Mỗi ngày bạn chỉ dự được **một** sự kiện. Trả về **số sự kiện tối đa** có thể tham dự.

**Ví dụ 1:** `events = [[1,2],[2,3],[3,4]]` → `3` (ngày 1 dự sự kiện 1, ngày 2 sự kiện 2, ngày 3 sự kiện 3).

**Ví dụ 2:** `events = [[1,2],[2,3],[3,4],[1,2]]` → `4`.

**Ràng buộc:**
- `1 <= events.length <= 10^5`
- `events[i].length == 2`
- `1 <= startDay_i <= endDay_i <= 10^5`

In [1]:
import heapq
import random
import time
from functools import lru_cache

examples = [
    ([[1, 2], [2, 3], [3, 4]], 3),
    ([[1, 2], [2, 3], [3, 4], [1, 2]], 4),
]
for ev, want in examples:
    print(f"events={ev} -> mong đợi {want}")

events=[[1, 2], [2, 3], [3, 4]] -> mong đợi 3
events=[[1, 2], [2, 3], [3, 4], [1, 2]] -> mong đợi 4


## 3. Kiến thức nền

### 3.1 Sắp xếp list các cặp và `key`

`sorted(events)` sắp theo phần tử đầu (start), hòa thì theo phần tử thứ hai. `sorted(events, key=lambda e: e[1])` sắp theo ngày kết thúc.

In [2]:
ev = [[3, 4], [1, 2], [2, 3], [1, 1]]
print("theo start:", sorted(ev))
print("theo end  :", sorted(ev, key=lambda e: e[1]))

theo start: [[1, 1], [1, 2], [2, 3], [3, 4]]
theo end  : [[1, 1], [1, 2], [2, 3], [3, 4]]


### 3.2 `heapq` — hàng đợi ưu tiên (min-heap)

`heapq` biến một list thành min-heap: `heappush(h, x)` thêm, `heappop(h)` lấy **phần tử nhỏ nhất**, `h[0]` xem phần tử nhỏ nhất. Mỗi thao tác `O(log n)`.

In [3]:
h = []
for x in [5, 2, 8, 2, 7]:
    heapq.heappush(h, x)
print("nhỏ nhất:", h[0])
print("lấy lần lượt:", [heapq.heappop(h) for _ in range(len(h))])

nhỏ nhất: 2
lấy lần lượt: [2, 2, 5, 7, 8]


### 3.3 Quét theo thời gian (sweep) với con trỏ

Sắp các sự kiện theo `start`, rồi cho "ngày hiện tại" `d` tăng dần; con trỏ `i` đẩy vào heap mọi sự kiện có `start ≤ d`. Mỗi sự kiện vào heap đúng một lần.

In [4]:
ev = sorted([[1, 2], [2, 3], [3, 4], [1, 2]])
i = 0
for d in range(1, 5):
    arrived = []
    while i < len(ev) and ev[i][0] <= d:
        arrived.append(ev[i]); i += 1
    print(f"ngày {d}: sự kiện mới mở = {arrived}")

ngày 1: sự kiện mới mở = [[1, 2], [1, 2]]
ngày 2: sự kiện mới mở = [[2, 3]]
ngày 3: sự kiện mới mở = [[3, 4]]
ngày 4: sự kiện mới mở = []


### 3.4 Union-Find (Disjoint Set) — tìm "ngày trống kế tiếp"

Mảng `parent` với `find(x)` trả về **ngày trống nhỏ nhất ≥ x**. Khi dùng ngày `x`, ta gán `parent[x] = x + 1` (ngày `x` "trỏ" sang ngày sau). Nén đường đi (path compression) làm các lần `find` sau gần như `O(1)`.

In [5]:
parent = list(range(8))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # nén đường đi (halving)
        x = parent[x]
    return x

for want in [2, 2, 2, 5, 4]:
    d = find(want)
    parent[d] = d + 1                   # dùng ngày d
    print(f"muốn ngày >= {want}: được ngày {d}")

muốn ngày >= 2: được ngày 2
muốn ngày >= 2: được ngày 3
muốn ngày >= 2: được ngày 4
muốn ngày >= 5: được ngày 5
muốn ngày >= 4: được ngày 6


### 3.5 Đệ quy có ghi nhớ (`lru_cache`) và bitmask

Dùng cho brute force: trạng thái "đã xét `i` sự kiện, tập ngày đã dùng là `mask`". `mask >> d & 1` kiểm tra ngày `d` đã dùng chưa, `mask | (1 << d)` đánh dấu ngày `d`.

In [6]:
mask = 0
mask |= 1 << 3
mask |= 1 << 1
print(bin(mask), "ngày 3 dùng?", mask >> 3 & 1, "| ngày 2 dùng?", mask >> 2 & 1)

0b1010 ngày 3 dùng? 1 | ngày 2 dùng? 0


## 4. Ý tưởng nền tảng / trực giác

Thử tính tay `events = [[1,3],[1,3],[2,2]]` (tự dựng — đáp án 3).

**Cách nghĩ ngây thơ — dự sớm nhất có thể theo thứ tự start:** `[1,3]` → ngày 1, `[1,3]` → ngày 2, `[2,2]` → ngày 2 đã bận ⇒ bỏ ⇒ chỉ **2** sự kiện. Sai!

**Cách nghĩ đúng — mỗi ngày, trong các sự kiện đang mở, dự cái "sắp hết hạn" nhất:**

| Ngày | Đang mở (end) | Chọn | Lý do |
|---|---|---|---|
| 1 | [1,3]→3, [1,3]→3 | một [1,3] | end nhỏ nhất (3) |
| 2 | [1,3]→3, [2,2]→2 | [2,2] | end = 2 gấp hơn |
| 3 | [1,3]→3 | [1,3] | |

⇒ **3** sự kiện ✔. Trực giác: sự kiện kết thúc sớm có ít "cơ hội" hơn — để dành sự kiện kết thúc muộn cho các ngày sau.

In [7]:
def greedy_by_start_naive(events):
    """Tham lam ngây thơ (SAI): duyệt theo start, dự ngày sớm nhất chưa dùng ≥ start."""
    used = set()
    cnt = 0
    for s, e in sorted(events):
        d = s
        while d in used:
            d += 1
        if d <= e:
            used.add(d); cnt += 1
    return cnt

ev = [[1, 3], [1, 3], [2, 2]]
print("tham lam theo start:", greedy_by_start_naive(ev), "(sai, đáp án đúng là 3)")

tham lam theo start: 2 (sai, đáp án đúng là 3)


## 5. Cách 1 — Brute Force / baseline: backtracking thử mọi cách

**Ý tưởng:** xét sự kiện theo thứ tự; với sự kiện `i`, hoặc **bỏ qua**, hoặc dự vào **một ngày còn trống** trong `[s, e]`. Lấy max. Dùng `lru_cache` trên `(i, mask)` để tránh lặp trạng thái. Chỉ dùng cho input **nhỏ** (để kiểm chứng).

In [8]:
def max_events_brute(events):
    ev = [tuple(e) for e in events]
    n = len(ev)

    @lru_cache(maxsize=None)
    def dfs(i, mask):
        if i == n:
            return 0
        best = dfs(i + 1, mask)                  # bỏ qua sự kiện i
        s, e = ev[i]
        for d in range(s, e + 1):
            if not (mask >> d) & 1:              # ngày d còn trống
                best = max(best, 1 + dfs(i + 1, mask | (1 << d)))
        return best

    return dfs(0, 0)

for ev, want in examples + [([[1, 3], [1, 3], [2, 2]], 3)]:
    print(f"{ev} -> {max_events_brute(ev)} (mong đợi {want})")

[[1, 2], [2, 3], [3, 4]] -> 3 (mong đợi 3)
[[1, 2], [2, 3], [3, 4], [1, 2]] -> 4 (mong đợi 4)
[[1, 3], [1, 3], [2, 2]] -> 3 (mong đợi 3)


**Độ phức tạp:** số trạng thái tới `n · 2^D` (`D` = ngày lớn nhất) ⇒ **mũ**. Chỉ dùng được khi `n`, `D` ≤ ~12. Với `10^5` sự kiện và `10^5` ngày thì không thể.

**Nút thắt:** ta thử **mọi** ngày cho mọi sự kiện. Cần một quy tắc chọn **không bao giờ hối hận** — tham lam.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tham lam theo ngày):** đi qua từng ngày `d = 1, 2, …`. Trong các sự kiện **đang mở** (`start ≤ d ≤ end`), dự sự kiện có **`end` nhỏ nhất**.

**Lập luận đổi chỗ:** giả sử một lịch tối ưu `OPT` vào ngày `d` dự sự kiện `y` (hoặc bỏ trống), còn tham lam chọn `x` với `end_x ≤ end_y`.
- Nếu `OPT` không dùng `x` ở đâu cả: thay `y` bằng `x` vào ngày `d` — số sự kiện không giảm.
- Nếu `OPT` dùng `x` ở ngày `d' > d`: đổi chỗ, `x` vào ngày `d`, `y` vào ngày `d'`. Hợp lệ vì `d < d' ≤ end_x ≤ end_y` và `start_y ≤ d < d'`.

⇒ Luôn tồn tại lời giải tối ưu đồng ý với lựa chọn của tham lam ở mỗi ngày.

**Quy tắc nếu… thì…:**
- Nếu sự kiện có `start ≤ d` thì đưa `end` của nó vào min-heap.
- Nếu đỉnh heap có `end < d` thì đã hết hạn ⇒ loại bỏ.
- Nếu heap còn phần tử thì pop đỉnh, `ans += 1`.

**Cách nhìn tương đương:** sắp theo `end` tăng dần, mỗi sự kiện chọn **ngày trống sớm nhất ≥ start** (nếu ≤ end). Cũng đúng bằng lập luận đổi chỗ tương tự.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp theo `end`, gán ngày trống sớm nhất (duyệt tuyến tính)

**Giải thích:** duyệt sự kiện theo `end` tăng dần; với mỗi sự kiện, tìm ngày `d` nhỏ nhất `≥ start` chưa bị dùng bằng cách tăng dần `d`. Nếu `d ≤ end` thì dự.

Đúng nhưng chậm khi nhiều sự kiện cùng đoạn dài: mỗi lần tìm có thể quét tới `O(D)` ngày.

In [9]:
def max_events_end_linear(events):
    used = set()
    ans = 0
    for s, e in sorted(events, key=lambda x: x[1]):
        d = s
        while d <= e and d in used:      # tìm ngày trống sớm nhất trong [s, e]
            d += 1
        if d <= e:
            used.add(d)
            ans += 1
    return ans

for ev, want in examples + [([[1, 3], [1, 3], [2, 2]], 3)]:
    print(f"{ev} -> {max_events_end_linear(ev)} (mong đợi {want})")

[[1, 2], [2, 3], [3, 4]] -> 3 (mong đợi 3)
[[1, 2], [2, 3], [3, 4], [1, 2]] -> 4 (mong đợi 4)
[[1, 3], [1, 3], [2, 2]] -> 3 (mong đợi 3)


**Độ phức tạp:** `O(n log n + n · D)` worst (ví dụ `10^5` sự kiện đều là `[1, 10^5]` ⇒ ~`5·10^9` bước). Bộ nhớ `O(n)`.

### 7.2 Cách 3 — Sắp theo `end` + Union-Find "ngày trống kế tiếp"

**Giải thích từng đoạn:**
1. `parent[d] = d` cho `d ∈ [0, maxEnd + 1]`; `find(d)` trả về ngày trống nhỏ nhất `≥ d`.
2. Với mỗi sự kiện (theo `end` tăng): `d = find(start)`. Nếu `d ≤ end` ⇒ dự, rồi `parent[d] = d + 1`.
3. Nén đường đi làm mỗi `find` gần như hằng số.

In [10]:
def max_events_dsu(events):
    max_end = max(e for _, e in events)
    parent = list(range(max_end + 2))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    ans = 0
    for s, e in sorted(events, key=lambda x: x[1]):
        d = find(s)
        if d <= e:
            ans += 1
            parent[d] = d + 1            # ngày d đã dùng -> trỏ sang ngày sau
    return ans

for ev, want in examples + [([[1, 3], [1, 3], [2, 2]], 3)]:
    print(f"{ev} -> {max_events_dsu(ev)} (mong đợi {want})")

[[1, 2], [2, 3], [3, 4]] -> 3 (mong đợi 3)
[[1, 2], [2, 3], [3, 4], [1, 2]] -> 4 (mong đợi 4)
[[1, 3], [1, 3], [2, 2]] -> 3 (mong đợi 3)


Code nộp LeetCode (Cách 3):

In [11]:
class Solution:
    def maxEvents(self, events: list[list[int]]) -> int:
        max_end = max(e for _, e in events)
        parent = list(range(max_end + 2))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        ans = 0
        for s, e in sorted(events, key=lambda x: x[1]):
            d = find(s)
            if d <= e:
                ans += 1
                parent[d] = d + 1
        return ans

print(Solution().maxEvents([[1, 2], [2, 3], [3, 4], [1, 2]]))

4


**Độ phức tạp:** `O(n log n + (n + D) · α)` ≈ `O(n log n + D)`. Bộ nhớ `O(D)`.

### 7.3 Cách 4 — Quét theo ngày + min-heap các `end`

**Giải thích từng đoạn:**
1. Sắp sự kiện theo `start`.
2. Cho ngày `d` chạy từ ngày nhỏ nhất. Đẩy `end` của mọi sự kiện có `start ≤ d` vào heap.
3. Loại các `end < d` (hết hạn) khỏi đỉnh heap.
4. Nếu heap không rỗng: pop (dự sự kiện kết thúc sớm nhất), `ans += 1`.
5. Tăng `d`. Nếu heap rỗng, **nhảy** thẳng `d` tới `start` của sự kiện kế tiếp (tránh quét ngày trống vô ích).

In [12]:
def max_events_heap(events):
    events = sorted(events)
    n = len(events)
    h = []
    i = ans = 0
    d = events[0][0]
    while i < n or h:
        if not h:
            d = max(d, events[i][0])            # nhảy qua các ngày không có sự kiện
        while i < n and events[i][0] <= d:
            heapq.heappush(h, events[i][1])
            i += 1
        while h and h[0] < d:                   # bỏ sự kiện đã hết hạn
            heapq.heappop(h)
        if h:
            heapq.heappop(h)                    # dự sự kiện kết thúc sớm nhất
            ans += 1
        d += 1
    return ans

for ev, want in examples + [([[1, 3], [1, 3], [2, 2]], 3)]:
    print(f"{ev} -> {max_events_heap(ev)} (mong đợi {want})")

[[1, 2], [2, 3], [3, 4]] -> 3 (mong đợi 3)
[[1, 2], [2, 3], [3, 4], [1, 2]] -> 4 (mong đợi 4)
[[1, 3], [1, 3], [2, 2]] -> 3 (mong đợi 3)


Code nộp LeetCode (Cách 4):

In [13]:
class Solution:
    def maxEvents(self, events: list[list[int]]) -> int:
        events.sort()
        n = len(events)
        h = []
        i = ans = 0
        d = events[0][0]
        while i < n or h:
            if not h:
                d = max(d, events[i][0])
            while i < n and events[i][0] <= d:
                heapq.heappush(h, events[i][1])
                i += 1
            while h and h[0] < d:
                heapq.heappop(h)
            if h:
                heapq.heappop(h)
                ans += 1
            d += 1
        return ans

print(Solution().maxEvents([[1, 2], [2, 3], [3, 4]]))

3


**Độ phức tạp:** sắp xếp `O(n log n)`; mỗi sự kiện push/pop heap đúng một lần `O(n log n)`; số ngày được duyệt ≤ `n + (số ngày có heap không rỗng)` ≤ `O(n + D)`. Tổng `O(n log n + D)`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 trên `events = [[1,3],[1,3],[2,2],[5,6],[5,5]]` (tự dựng). Bảng trạng thái:

| Ngày | Đẩy vào heap | Bỏ hết hạn | Heap trước khi chọn | Dự (end) | ans |
|---|---|---|---|---|---|
| 1 | 3, 3 | – | [3, 3] | 3 | 1 |
| 2 | 2 | – | [2, 3] | 2 | 2 |
| 3 | – | – | [3] | 3 | 3 |
| 4 | – | – | [] ⇒ nhảy tới ngày 5 | | |
| 5 | 6, 5 | – | [5, 6] | 5 | 4 |
| 6 | – | – | [6] | 6 | 5 |

In [14]:
def trace_heap(events):
    events = sorted(events)
    n = len(events)
    h = []
    i = ans = 0
    d = events[0][0]
    while i < n or h:
        if not h and d < events[i][0]:
            print(f"  heap rỗng -> nhảy từ ngày {d} tới ngày {events[i][0]}")
            d = events[i][0]
        pushed = []
        while i < n and events[i][0] <= d:
            heapq.heappush(h, events[i][1]); pushed.append(events[i]); i += 1
        dropped = []
        while h and h[0] < d:
            dropped.append(heapq.heappop(h))
        before = sorted(h)
        took = None
        if h:
            took = heapq.heappop(h); ans += 1
        print(f"ngày {d}: đẩy {pushed}, bỏ {dropped}, heap={before}, dự end={took}, ans={ans}")
        d += 1
    return ans

print("Kết quả:", trace_heap([[1, 3], [1, 3], [2, 2], [5, 6], [5, 5]]))

ngày 1: đẩy [[1, 3], [1, 3]], bỏ [], heap=[3, 3], dự end=3, ans=1
ngày 2: đẩy [[2, 2]], bỏ [], heap=[2, 3], dự end=2, ans=2
ngày 3: đẩy [], bỏ [], heap=[3], dự end=3, ans=3
  heap rỗng -> nhảy từ ngày 4 tới ngày 5
ngày 5: đẩy [[5, 5], [5, 6]], bỏ [], heap=[5, 6], dự end=5, ans=4
ngày 6: đẩy [], bỏ [], heap=[6], dự end=6, ans=5
Kết quả: 5


Trace Cách 3 (Union-Find) trên cùng input — in ngày được gán cho từng sự kiện:

In [15]:
def trace_dsu(events):
    max_end = max(e for _, e in events)
    parent = list(range(max_end + 2))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    ans = 0
    for s, e in sorted(events, key=lambda x: x[1]):
        d = find(s)
        if d <= e:
            ans += 1
            parent[d] = d + 1
            print(f"sự kiện [{s},{e}] -> ngày {d}  (ans={ans})")
        else:
            print(f"sự kiện [{s},{e}] -> ngày trống sớm nhất là {d} > {e} -> bỏ")
    return ans

print("Kết quả:", trace_dsu([[1, 3], [1, 3], [2, 2], [5, 6], [5, 5]]))

sự kiện [2,2] -> ngày 2  (ans=1)
sự kiện [1,3] -> ngày 1  (ans=2)
sự kiện [1,3] -> ngày 3  (ans=3)
sự kiện [5,5] -> ngày 5  (ans=4)
sự kiện [5,6] -> ngày 6  (ans=5)
Kết quả: 5


## 9. Code nộp LeetCode

Chọn Cách 4 (quét ngày + min-heap) — lời giải kinh điển, đúng chữ ký.

In [16]:
import heapq

class Solution:
    def maxEvents(self, events: list[list[int]]) -> int:
        events.sort()                       # theo ngày bắt đầu
        n = len(events)
        h = []                              # min-heap các ngày kết thúc của sự kiện đang mở
        i = ans = 0
        d = events[0][0]
        while i < n or h:
            if not h:
                d = max(d, events[i][0])    # nhảy qua ngày trống
            while i < n and events[i][0] <= d:
                heapq.heappush(h, events[i][1])
                i += 1
            while h and h[0] < d:           # hết hạn
                heapq.heappop(h)
            if h:
                heapq.heappop(h)            # dự sự kiện kết thúc sớm nhất
                ans += 1
            d += 1
        return ans

for ev, want in examples:
    got = Solution().maxEvents([e[:] for e in ev])
    assert got == want
    print(f"maxEvents({ev}) = {got}")

maxEvents([[1, 2], [2, 3], [3, 4]]) = 3
maxEvents([[1, 2], [2, 3], [3, 4], [1, 2]]) = 4


## 10. Vì sao thuật toán đúng

**Bất biến của Cách 4:** khi xử lý ngày `d`, heap chứa đúng `end` của các sự kiện **đã mở** (`start ≤ d`) và **chưa được dự**, sau khi loại các cái hết hạn thì mọi phần tử đều có `end ≥ d` — tức là **có thể dự vào ngày `d`**.

**Tối ưu (đổi chỗ):** gọi `G` là lịch tham lam, `OPT` là một lịch tối ưu giống `G` ở mọi ngày `< d`. Ở ngày `d`, `G` chọn `x` có `end` nhỏ nhất.
1. `OPT` bỏ trống ngày `d`: nếu `OPT` dùng `x` ở ngày `d' > d` thì chuyển `x` sang ngày `d`; nếu không dùng `x` thì thêm `x` vào ngày `d` (mâu thuẫn tính tối ưu, nên trường hợp này chỉ có thể là chuyển). Số sự kiện không giảm.
2. `OPT` dự `y ≠ x` ở ngày `d`: nếu `x` không có trong `OPT`, thay `y` bằng `x`. Nếu `x` ở ngày `d' > d`, hoán đổi: `y` sang `d'` hợp lệ vì `start_y ≤ d < d' ≤ end_x ≤ end_y`.

Sau mỗi bước, `OPT` mới vẫn tối ưu và giống `G` thêm một ngày ⇒ quy nạp: `|G| = |OPT|`.

**Nhảy ngày khi heap rỗng** không bỏ sót gì vì những ngày đó không có sự kiện nào mở. **Cách 3** đúng theo lập luận đổi chỗ tương tự cho thứ tự `end` tăng dần, và Union-Find chỉ là cách tìm "ngày trống sớm nhất ≥ start" nhanh hơn Cách 2.

## 11. Phân tích Time / Space

Gọi `n` = số sự kiện, `D` = ngày kết thúc lớn nhất (≤ `10^5`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Backtracking + memo | `O(n · 2^D · D)` | `O(n · 2^D)` | chỉ để kiểm chứng |
| 2. Sắp theo end + quét ngày tuyến tính | `O(n log n + n·D)` worst | `O(n)` | đúng nhưng có thể TLE |
| 3. Sắp theo end + Union-Find | `O(n log n + D·α)` | `O(D)` | rất nhanh |
| 4. Quét ngày + heap | `O(n log n + D)` | `O(n)` | lời giải chuẩn |

Trường hợp xấu nhất của Cách 2: mọi sự kiện là `[1, 10^5]`. Cách 4 với input thưa (ngày rời rạc lớn) nhờ "nhảy ngày" chỉ duyệt các ngày có sự kiện.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
METHODS = {
    "brute": max_events_brute,
    "end_linear": max_events_end_linear,
    "dsu": max_events_dsu,
    "heap": max_events_heap,
    "submit": lambda ev: Solution().maxEvents([e[:] for e in ev]),
}
cases = examples + [
    ([[1, 1]], 1),                                   # 1 sự kiện 1 ngày
    ([[1, 1], [1, 1], [1, 1]], 1),                   # trùng hết vào 1 ngày
    ([[1, 3], [1, 3], [2, 2]], 3),                   # phản ví dụ tham lam theo start
    ([[1, 5], [1, 5], [1, 5], [2, 3], [2, 3]], 5),
    ([[1, 2], [1, 2], [1, 2], [1, 2]], 2),           # nhiều hơn số ngày
    ([[1, 10], [2, 2], [2, 2], [2, 2], [2, 2]], 2),
    ([[5, 5], [1, 1], [3, 3]], 3),                   # ngày rời rạc
    ([[1, 4], [4, 4], [2, 2], [3, 4], [1, 1]], 4),
]
for ev, want in cases:
    for name, f in METHODS.items():
        got = f(ev)
        assert got == want, (name, ev, got, want)
print(f"{len(cases)} ca × {len(METHODS)} cách: đúng hết")

10 ca × 5 cách: đúng hết


### 12.2 Đối chiếu ngẫu nhiên với backtracking và ghép cặp hai phía (seed cố định)

Thêm một "trọng tài" thứ hai: **ghép cặp cực đại** giữa sự kiện và ngày (thuật toán Kuhn — tìm đường tăng). Bài toán chính là ghép cặp cực đại trên đồ thị hai phía "sự kiện — ngày nằm trong đoạn".

In [18]:
def max_events_matching(events):
    """Ghép cặp cực đại (Kuhn): sự kiện -> ngày. O(n * tổng độ dài đoạn)."""
    match_day = {}                       # ngày -> chỉ số sự kiện

    def try_assign(i, seen):
        s, e = events[i]
        for d in range(s, e + 1):
            if d in seen:
                continue
            seen.add(d)
            if d not in match_day or try_assign(match_day[d], seen):
                match_day[d] = i
                return True
        return False

    return sum(try_assign(i, set()) for i in range(len(events)))

rng = random.Random(1353)
naive_wrong = 0
for _ in range(400):
    n = rng.randint(1, 7)
    D = rng.randint(1, 8)
    ev = []
    for _ in range(n):
        s = rng.randint(1, D); e = rng.randint(s, D)
        ev.append([s, e])
    ref = max_events_brute(ev)
    assert max_events_matching(ev) == ref
    for name, f in METHODS.items():
        assert f(ev) == ref, (name, ev)
    naive_wrong += greedy_by_start_naive(ev) != ref
print("400 bộ nhỏ: mọi cách khớp backtracking và ghép cặp")
print("Tham lam theo start (sai) trượt", naive_wrong, "/ 400 bộ")

400 bộ nhỏ: mọi cách khớp backtracking và ghép cặp
Tham lam theo start (sai) trượt 8 / 400 bộ


In [19]:
# Bộ vừa (n tới 60, ngày tới 40): so với ghép cặp
rng = random.Random(7)
for _ in range(200):
    n = rng.randint(1, 60); D = rng.randint(1, 40)
    ev = []
    for _ in range(n):
        s = rng.randint(1, D); e = rng.randint(s, min(D, s + rng.randint(0, 10)))
        ev.append([s, e])
    ref = max_events_matching(ev)
    for name in ["end_linear", "dsu", "heap", "submit"]:
        assert METHODS[name](ev) == ref, (name, ev)
print("200 bộ vừa: khớp ghép cặp cực đại")

200 bộ vừa: khớp ghép cặp cực đại


### 12.3 Kích thước lớn nhất `n = 10^5`, `D = 10^5`

In [20]:
rng = random.Random(99)
big_random = []
for _ in range(10**5):
    s = rng.randint(1, 10**5); e = rng.randint(s, min(10**5, s + 1000))
    big_random.append([s, e])
big_same = [[1, 10**5]] * 10**5           # trường hợp xấu cho Cách 2
for label, ev in [("ngẫu nhiên", big_random), ("toàn [1,1e5]", big_same)]:
    res = []
    for name in ["dsu", "heap"]:
        st = time.perf_counter()
        res.append(METHODS[name](ev))
        print(f"{label:13} {name:5} -> {res[-1]}  ({time.perf_counter() - st:.3f}s)")
    assert res[0] == res[1]

ngẫu nhiên    dsu   -> 99788  (0.090s)


ngẫu nhiên    heap  -> 99788  (0.134s)
toàn [1,1e5]  dsu   -> 100000  (0.031s)
toàn [1,1e5]  heap  -> 100000  (0.037s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam theo ngày bắt đầu (gợi ý "dễ dãi")

In [21]:
ev = [[1, 3], [1, 3], [2, 2]]
print("tham lam theo start:", greedy_by_start_naive(ev), "| đúng:", max_events_heap(ev))
ev2 = [[1, 4], [1, 4], [1, 4], [2, 2], [3, 3]]
print("tham lam theo start:", greedy_by_start_naive(ev2), "| đúng:", max_events_heap(ev2))

tham lam theo start: 2 | đúng: 3
tham lam theo start: 3 | đúng: 4


**Vì sao sai:** sự kiện bắt đầu sớm chưa chắc gấp — `[1,3]` còn nhiều ngày, còn `[2,2]` chỉ có đúng ngày 2. Tham lam theo start dùng hết ngày 2 cho `[1,3]`. **Sửa:** mỗi ngày chọn sự kiện có **end** nhỏ nhất (heap), hoặc duyệt theo end tăng dần.

### Lỗi 2 — Quên loại các sự kiện đã hết hạn khỏi heap

In [22]:
def wrong_no_expire(events):
    events = sorted(events)
    h, i, ans, d, n = [], 0, 0, events[0][0], len(events)
    while i < n or h:
        if not h:
            d = max(d, events[i][0])
        while i < n and events[i][0] <= d:
            heapq.heappush(h, events[i][1]); i += 1
        if h:                         # không kiểm tra h[0] < d
            heapq.heappop(h); ans += 1
        d += 1
    return ans

ev = [[1, 1], [1, 1], [1, 1]]
print("sai:", wrong_no_expire(ev), "| đúng:", max_events_heap(ev))

sai: 3 | đúng: 1


**Vì sao sai:** sự kiện `[1,1]` vẫn nằm trong heap ở ngày 2, 3 và bị "dự" dù đã kết thúc. **Sửa:** trước khi pop, loại mọi `h[0] < d`.

### Lỗi 3 — Heap chứa `start` thay vì `end`

In [23]:
def wrong_heap_start(events):
    events = sorted(events)
    h, i, ans, d, n = [], 0, 0, events[0][0], len(events)
    while i < n or h:
        if not h:
            d = max(d, events[i][0])
        while i < n and events[i][0] <= d:
            heapq.heappush(h, (events[i][0], events[i][1])); i += 1   # ưu tiên theo start
        while h and h[0][1] < d:
            heapq.heappop(h)
        if h:
            heapq.heappop(h); ans += 1
        d += 1
    return ans

ev = [[1, 3], [1, 3], [2, 2]]
print("sai:", wrong_heap_start(ev), "| đúng:", max_events_heap(ev))

sai: 2 | đúng: 3


**Vì sao sai:** ưu tiên theo `start` chẳng khác gì Lỗi 1, và việc chỉ kiểm tra hết hạn ở đỉnh heap còn để sót phần tử hết hạn bên dưới. **Sửa:** heap theo **end**.

### Lỗi 4 — Chỉ quét ngày tới `max(start)` (bỏ sót ngày cuối)

In [24]:
def wrong_day_limit(events):
    events = sorted(events)
    h, i, ans, n = [], 0, 0, len(events)
    for d in range(1, max(s for s, _ in events) + 1):     # thiếu các ngày sau max start
        while i < n and events[i][0] <= d:
            heapq.heappush(h, events[i][1]); i += 1
        while h and h[0] < d:
            heapq.heappop(h)
        if h:
            heapq.heappop(h); ans += 1
    return ans

ev = [[1, 5], [1, 5], [1, 5]]
print("sai:", wrong_day_limit(ev), "| đúng:", max_events_heap(ev))

sai: 1 | đúng: 3


**Vì sao sai:** sau ngày bắt đầu cuối cùng, heap vẫn có thể còn sự kiện dự được. **Sửa:** lặp tới khi **cả** danh sách và heap đều rỗng (`while i < n or h`), hoặc tới `max(end)`.

### Lỗi 5 — Đếm số ngày bị phủ thay vì số sự kiện

In [25]:
def wrong_cover(events):
    days = set()
    for s, e in events:
        days.update(range(s, e + 1))
    return min(len(days), len(events))

ev = [[1, 1], [1, 1], [2, 3]]
print("sai:", wrong_cover(ev), "| đúng:", max_events_heap(ev))
ev = [[1, 4], [1, 1], [1, 1], [1, 1]]
print("sai:", wrong_cover(ev), "| đúng:", max_events_heap(ev))

sai: 3 | đúng: 2
sai: 4 | đúng: 2


**Vì sao sai:** `min(#ngày, #sự kiện)` chỉ là **cận trên**; ở ví dụ hai, 4 ngày và 4 sự kiện nhưng ba sự kiện cùng tranh ngày 1. **Sửa:** phải ghép cặp thực sự (heap/Union-Find).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mỗi việc chọn một thời điểm trong khoảng, mỗi thời điểm một việc, tối đa số việc"** ⇒ tham lam **deadline sớm nhất trước** (Earliest Deadline First) + heap.
- Heap chứa **thứ cần ưu tiên** (ở đây là `end`); trước khi lấy, **dọn phần tử hết hạn**.
- Khi "tìm chỗ trống kế tiếp" lặp đi lặp lại ⇒ **Union-Find next-free** (cũng dùng trong bài đặt phòng, cấp số ghế, 1488...).
- Kiểm chứng tham lam bằng brute force trên input nhỏ — rất nhiều tham lam "nghe hợp lý" là sai (Lỗi 1).
- So sánh với bài **435/646 (chọn khoảng không giao nhau)**: ở đó mỗi khoảng chiếm **cả đoạn**; ở đây mỗi sự kiện chỉ chiếm **một ngày**.

## 15. Biến thể của bài

1. **LeetCode 1751 — Maximum Number of Events That Can Be Attended II:** mỗi sự kiện có giá trị và chiếm **cả đoạn**, dự tối đa `k` sự kiện ⇒ DP + nhị phân.
2. **Mỗi ngày dự được tối đa `c` sự kiện** ⇒ heap như cũ, mỗi ngày pop tới `c` lần.
3. **Trả về lịch cụ thể** (sự kiện nào ngày nào) — Bài 5.
4. **LeetCode 630 — Course Schedule III:** mỗi khóa học có thời lượng và hạn chót ⇒ sắp theo hạn + max-heap thời lượng.
5. **Sự kiện có trọng số, mỗi sự kiện một ngày** ⇒ ghép cặp có trọng số / tham lam theo giá trị với Union-Find (bài "job sequencing with deadlines").

In [26]:
def max_events_capacity(events, c):
    """Biến thể 2: mỗi ngày dự tối đa c sự kiện."""
    events = sorted(events)
    h, i, ans, n = [], 0, 0, len(events)
    d = events[0][0]
    while i < n or h:
        if not h:
            d = max(d, events[i][0])
        while i < n and events[i][0] <= d:
            heapq.heappush(h, events[i][1]); i += 1
        while h and h[0] < d:
            heapq.heappop(h)
        for _ in range(c):
            if not h:
                break
            heapq.heappop(h); ans += 1
        d += 1
    return ans

print("c=1:", max_events_capacity([[1, 1], [1, 1], [1, 2]], 1), "| c=2:", max_events_capacity([[1, 1], [1, 1], [1, 2]], 2))

c=1: 2 | c=2: 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Backtracking + memo | thử mọi ngày cho mọi sự kiện | mũ | mũ | kiểm chứng n, D nhỏ |
| (trọng tài) Ghép cặp Kuhn | đồ thị hai phía sự kiện–ngày | `O(n · Σ độ dài)` | `O(D)` | kiểm chứng cỡ vừa |
| 2. Theo end + quét tuyến tính | ngày trống sớm nhất ≥ start | `O(n log n + nD)` | `O(n)` | đơn giản, input nhỏ |
| 3. Theo end + Union-Find | next-free day | `O(n log n + D)` | `O(D)` | nhanh, code ngắn |
| 4. Quét ngày + heap end | EDF mỗi ngày | `O(n log n + D)` | `O(n)` | **chuẩn phỏng vấn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maxEvents([[1,4],[4,4],[2,2],[3,4],[1,1]])` và `maxEvents([[1,100000]])`?

**Bài 2 (tính tay):** Chạy tay Cách 4 trên `[[1,2],[1,2],[3,3],[1,5],[1,5]]`: ghi heap mỗi ngày.

**Bài 3 (điền TODO):** Hoàn thiện hàm `find` của Union-Find "ngày trống kế tiếp" có **nén đường đi đệ quy**.

**Bài 4 (tự viết lại):** Viết lại Cách 4 **không** dùng mẹo "nhảy ngày", lặp `d` từ 1 tới `max(end)`.

**Bài 5 (mini challenge):** Viết `schedule(events)` trả về list `day[i]` (ngày dự sự kiện `i`, hoặc `-1` nếu không dự) sao cho số sự kiện được dự là tối đa. Viết thêm hàm kiểm tra tính hợp lệ.

In [27]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def make_find(parent):
    def find(x):
        # TODO: nếu parent[x] == x thì trả x
        # TODO: ngược lại parent[x] = find(parent[x]) rồi trả parent[x]
        pass
    return find

In [28]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def max_events_no_jump(events):
    # TODO: for d in range(1, max_end + 1): đẩy, dọn hết hạn, pop
    pass

def schedule(events):
    # TODO: heap chứa (end, index) để biết sự kiện nào được dự vào ngày nào
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Có 5 sự kiện và các ngày 1..4. (2) Không thể quá 4. (3) Một sự kiện luôn dự được.

**Bài 2:** (1) Ngày 1 heap `[2,2,5,5]`. (2) Ngày 2 heap `[2,5,5]`; ngày 3 thêm `3`. (3) Đếm số lần pop thành công.

**Bài 3:** (1) Trường hợp cơ sở `parent[x] == x`. (2) Gán `parent[x] = find(parent[x])`. (3) Cẩn thận giới hạn đệ quy với `D` lớn (dùng vòng lặp trong code nộp).

**Bài 4:** (1) `max_end = max(e for _, e in events)`. (2) Vòng `for d in range(1, max_end + 1)`. (3) Thân vòng lặp giống Cách 4 bỏ phần nhảy.

**Bài 5:** (1) Heap chứa cặp `(end, i)`. (2) Khi pop được `(end, i)` ở ngày `d` thì `day[i] = d`. (3) Hàm kiểm tra: mỗi `day[i]` (≠ −1) nằm trong `[s_i, e_i]`, các ngày đôi một khác nhau, và số sự kiện được dự bằng `maxEvents`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
print("Bài 1:", max_events_heap([[1, 4], [4, 4], [2, 2], [3, 4], [1, 1]]), max_events_heap([[1, 100000]]))
assert max_events_heap([[1, 4], [4, 4], [2, 2], [3, 4], [1, 1]]) == 4
assert max_events_heap([[1, 100000]]) == 1

# Bài 2
print("Bài 2:")
assert trace_heap([[1, 2], [1, 2], [3, 3], [1, 5], [1, 5]]) == 5

Bài 1: 4 1
Bài 2:
ngày 1: đẩy [[1, 2], [1, 2], [1, 5], [1, 5]], bỏ [], heap=[2, 2, 5, 5], dự end=2, ans=1
ngày 2: đẩy [], bỏ [], heap=[2, 5, 5], dự end=2, ans=2
ngày 3: đẩy [[3, 3]], bỏ [], heap=[3, 5, 5], dự end=3, ans=3
ngày 4: đẩy [], bỏ [], heap=[5, 5], dự end=5, ans=4
ngày 5: đẩy [], bỏ [], heap=[5], dự end=5, ans=5


In [30]:
# Bài 3
def make_find_sol(parent):
    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    return find

parent = list(range(10))
find = make_find_sol(parent)
got = []
for want in [3, 3, 3, 1, 4]:
    d = find(want); parent[d] = d + 1; got.append(d)
print("Bài 3:", got)
assert got == [3, 4, 5, 1, 6]

# Bài 4
def max_events_no_jump_sol(events):
    events = sorted(events)
    max_end = max(e for _, e in events)
    h, i, ans, n = [], 0, 0, len(events)
    for d in range(1, max_end + 1):
        while i < n and events[i][0] <= d:
            heapq.heappush(h, events[i][1]); i += 1
        while h and h[0] < d:
            heapq.heappop(h)
        if h:
            heapq.heappop(h); ans += 1
    return ans

rng = random.Random(4)
for _ in range(300):
    ev = []
    for _ in range(rng.randint(1, 30)):
        s = rng.randint(1, 20); ev.append([s, rng.randint(s, 25)])
    assert max_events_no_jump_sol(ev) == max_events_heap(ev)
print("Bài 4: khớp Cách 4 trên 300 bộ")

Bài 3: [3, 4, 5, 1, 6]


Bài 4: khớp Cách 4 trên 300 bộ


In [31]:
# Bài 5
def schedule_sol(events):
    order = sorted(range(len(events)), key=lambda i: events[i][0])
    day = [-1] * len(events)
    h, p, n = [], 0, len(events)
    d = events[order[0]][0]
    while p < n or h:
        if not h:
            d = max(d, events[order[p]][0])
        while p < n and events[order[p]][0] <= d:
            i = order[p]; heapq.heappush(h, (events[i][1], i)); p += 1
        while h and h[0][0] < d:
            heapq.heappop(h)
        if h:
            _, i = heapq.heappop(h); day[i] = d
        d += 1
    return day

def is_valid_schedule(events, day):
    used = [d for d in day if d != -1]
    if len(used) != len(set(used)):
        return False
    return all(d == -1 or events[i][0] <= d <= events[i][1] for i, d in enumerate(day))

ev = [[1, 3], [1, 3], [2, 2], [5, 6], [5, 5]]
print("Bài 5:", schedule_sol(ev))
rng = random.Random(5)
for _ in range(300):
    ev = []
    for _ in range(rng.randint(1, 7)):
        s = rng.randint(1, 8); ev.append([s, rng.randint(s, 8)])
    day = schedule_sol(ev)
    assert is_valid_schedule(ev, day)
    assert sum(d != -1 for d in day) == max_events_brute(ev)
print("Bài 5: lịch hợp lệ và tối ưu trên 300 bộ (so với backtracking)")

Bài 5: [1, 3, 2, 6, 5]
Bài 5: lịch hợp lệ và tối ưu trên 300 bộ (so với backtracking)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template quét ngày + heap (EDF):**

```python
events.sort()
h, i, ans, d = [], 0, 0, events[0][0]
while i < n or h:
    if not h: d = max(d, events[i][0])          # nhảy ngày trống
    while i < n and events[i][0] <= d:
        heappush(h, events[i][1]); i += 1      # mở sự kiện
    while h and h[0] < d: heappop(h)           # dọn hết hạn
    if h: heappop(h); ans += 1                 # dự cái gấp nhất
    d += 1
```

**Template Union-Find next-free:** sắp theo end; `d = find(s)`; nếu `d <= e`: `ans += 1; parent[d] = d + 1`.

**Checklist trước khi submit:**
- [ ] Ưu tiên theo **end** (không phải start).
- [ ] Dọn sự kiện hết hạn (`h[0] < d`) trước khi pop.
- [ ] Vòng lặp chạy tới khi hết sự kiện **và** heap rỗng.
- [ ] Thử phản ví dụ `[[1,3],[1,3],[2,2]]` ⇒ 3.
- [ ] Thử `[[1,1],[1,1],[1,1]]` ⇒ 1.
- [ ] Đúng chữ ký: `maxEvents(self, events: list[list[int]]) -> int`.